<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS13/datasetPBL/TugasPraktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load dan gabungkan dataset milikmu (pengganti mnist.load_data())
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')
df = pd.merge(df_reviews, df_cafes, on='cafe_id', how='left', suffixes=('_review', '_cafe'))

# Ekstraksi 784 fitur (28x28) dalam rentang [0, 1] seperti citra MNIST ternormalisasi
tfidf = TfidfVectorizer(max_features=28 * 28, ngram_range=(1, 2))
X_784 = tfidf.fit_transform((df['name'] + " " + df['text'].astype(str))).toarray()

if X_784.shape[1] < 28 * 28:
    X_784 = np.pad(X_784, ((0, 0), (0, (28 * 28) - X_784.shape[1])), mode='constant')

X_28x28 = X_784.reshape(-1, 28, 28)

# Encode target 5 kategori tempat usaha (categories)
le = LabelEncoder()
y_raw = le.fit_transform(df['categories'])
num_classes = len(le.classes_)

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X_28x28, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# One-hot encoding label
y_train = to_categorical(y_train_raw, num_classes)
y_test = to_categorical(y_test_raw, num_classes)

# 2. Bangun model JST (2 Hidden Layer: 128 dan 64, ReLU)
tf.random.set_seed(42)
model = Sequential([
    Flatten(input_shape=(28, 28)),              # Ubah matriks 28x28 menjadi vektor 784
    Dense(128, activation='relu'),              # Hidden layer 1
    Dense(64, activation='relu'),               # Hidden layer 2
    Dense(num_classes, activation='softmax')    # Output layer (5 kelas kategori)
])

# 3. Kompilasi model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 4. Latih model & ukur waktu pelatihan
start_time = time.time()
model.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)
waktu_dasar = time.time() - start_time

# 5. Evaluasi model
loss_dasar, acc_dasar = model.evaluate(X_test, y_test, verbose=0)
print(f"Model Dasar (128, 64 - ReLU) -> Loss: {loss_dasar:.4f} | Akurasi: {acc_dasar:.4f} | Waktu: {waktu_dasar:.2f} detik")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model Dasar (128, 64 - ReLU) -> Loss: 0.0212 | Akurasi: 0.9933 | Waktu: 5.35 detik


In [2]:
def eksperimen_jst(hidden_layers, activation='relu', nama_model=''):
    tf.random.set_seed(42)
    np.random.seed(42)

    m = Sequential([Flatten(input_shape=(28, 28))])
    for units in hidden_layers:
        m.add(Dense(units, activation=activation))
    m.add(Dense(num_classes, activation='softmax'))

    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    t0 = time.time()
    m.fit(X_train, y_train, epochs=15, batch_size=32, verbose=0)
    durasi = time.time() - t0

    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{nama_model:<32} -> Loss: {loss:.4f} | Akurasi: {acc:.4f} | Waktu: {durasi:.2f} detik")
    return loss, acc, durasi

# 1. Ubah jumlah neuron hidden layer menjadi (256, 128)
eksperimen_jst([256, 128], activation='relu', nama_model='1. Ubah Neuron (256, 128 - ReLU)')

# 2. Tambah 1 hidden layer lagi menjadi 3 hidden layer (256, 128, 64)
eksperimen_jst([256, 128, 64], activation='relu', nama_model='2. 3 Hidden Layer (256, 128, 64)')

# 3. Eksperimen fungsi aktivasi Sigmoid (128, 64)
eksperimen_jst([128, 64], activation='sigmoid', nama_model='3. Aktivasi Sigmoid (128, 64)')

1. Ubah Neuron (256, 128 - ReLU) -> Loss: 0.0204 | Akurasi: 0.9978 | Waktu: 8.10 detik
2. 3 Hidden Layer (256, 128, 64) -> Loss: 0.0169 | Akurasi: 0.9933 | Waktu: 7.78 detik
3. Aktivasi Sigmoid (128, 64)    -> Loss: 0.0526 | Akurasi: 0.9843 | Waktu: 5.64 detik


(0.05261031538248062, 0.9843049049377441, 5.6436767578125)